<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 8</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Consumer Groups & Scalability</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Learn how Kafka distributes load and scales horizontally using a real Python Consumer.</p>
</div>

---

## 🎯 Overview

Kafka is designed for massive scalability. A single application might not be fast enough to process all messages in a high-volume topic. 
By putting multiple consumers in the same **Consumer Group** (`group.id`), Kafka will automatically load-balance the partitions across them!

**Golden Rules of Consumer Groups:**
1. Messages in a single partition are read by only **one** consumer in a group.
2. The maximum number of active consumers in a group equals the **number of partitions**. (Any extra consumers will sit idle as hot standbys).
3. Consumers in **different** groups operate entirely independently (Publish/Subscribe pattern).

---

## ⚙️ Prerequisites

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Ensure any previous multi-broker clusters are stopped. Run <code>docker-compose down</code> in Lab7 before proceeding.
</div>

---

## <span style="color: #667eea;">Step 1:</span> Start the Kafka Cluster


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d

---

## <span style="color: #667eea;">Step 2:</span> Create a Multi-Partition Topic

To distribute load, we MUST have multiple partitions! We will create `group-topic` with **3 partitions**.

In [ ]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic group-topic \
  --partitions 3 \
  --replication-factor 1

---

## <span style="color: #667eea;">Step 3:</span> Review the Python Consumer

Instead of using the console tools, we are going to use the custom `consumer.py` script provided in this folder. 
Open the file and notice how the `group_id` is hardcoded to `group-A` directly in the code!

Let's make sure the required Kafka library is installed in our Jupyter environment.

In [ ]:
%pip install confluent-kafka

---

## <span style="color: #667eea;">Step 4:</span> Start 3 Background Consumers (Group A)

We will use Python's built-in `subprocess` module to start 3 instances of our script concurrently entirely within our Jupyter environment. 
Because `group.id` is hardcoded to `group-A` inside the file, they will all join the same group! We will direct their output into separate local log files (`c1.log`, `c2.log`, `c3.log`).

In [ ]:
import subprocess
import time

print("Starting 3 Python consumers in the background...")

p1 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c1.log', 'w'))
p2 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c2.log', 'w'))
p3 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c3.log', 'w'))

consumers = [p1, p2, p3]

# Wait a few seconds for them to connect to the broker and join the group
time.sleep(3)
print("Consumers are running!")

---

## <span style="color: #667eea;">Step 5:</span> Describe the Consumer Group

Let's look at what Kafka did under the hood using the `kafka-consumer-groups` tool.

You should see exactly 3 distinct `CLIENT-ID`s. Kafka has assigned Partition 0 to one consumer, Partition 1 to another, and Partition 2 to the third!

In [ ]:
!docker exec kafka kafka-consumer-groups \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A

---

## <span style="color: #667eea;">Step 6:</span> Produce Data and Observe Load Balancing

Now let's produce exactly **300 messages** into the topic.

In [ ]:
!docker exec kafka kafka-producer-perf-test \
  --topic group-topic \
  --num-records 300 \
  --record-size 100 \
  --throughput -1 \
  --producer-props bootstrap.servers=localhost:9092

Let's count the lines in the local log files! Because Kafka distributes messages across partitions (using a Round-Robin strategy when there are no Keys), our 3 consumers should have shared the workload evenly. 
You should see roughly **100 messages** processed in each log file!

In [ ]:
import time
time.sleep(2) # Give the python scripts a second to flush logs to disk

for f in ['c1.log', 'c2.log', 'c3.log']:
    with open(f, 'r') as file:
        # Count the number of lines (subtracting 1 for the startup 'joined group' message)
        count = sum(1 for line in file) - 1
        print(f"{f} processed {count} messages")

You can even peek inside the log file to see our Python consumer printing out the exact Partition and Offset it processed! Notice how we used the OS Process ID (`os.getpid()`) to dynamically name each script!

In [ ]:
with open('c1.log', 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

---

## <span style="color: #667eea;">Step 7:</span> The Idle Consumer (Over-provisioning)

What happens if we add a 4th consumer to `group-A`? Because there are only 3 partitions, the 4th consumer will sit idle. It is a "hot standby" and will only take over if one of the first 3 consumers crashes.

Let's start one more instance and describe the group again. Look closely at the table for the new consumer. It will be listed, but under `ASSIGNMENT`, it won't have any partitions!

In [ ]:
print("Starting idle consumer...")
p4 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c4.log', 'w'))
consumers.append(p4)

time.sleep(4)
!docker exec kafka kafka-consumer-groups --bootstrap-server localhost:9092 --describe --group group-A

---

## <span style="color: #667eea;">Step 8:</span> A Different Consumer Group

What if a totally different application (e.g., an Analytics service) needs to read the same data?
If they use a **different** group ID, they act completely independently. They will start from the beginning and get assigned *all* partitions.

### 👨‍💻 Interactive Code Change!
1. Open `consumer.py` in your editor.
2. Change `group_id = 'group-A'` to `group_id = 'group-B'`.
3. **Save the file.**
4. Run the cell below to launch the modified code in the background!

In [ ]:
print("Starting consumer in group-B...")
p5 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c5.log', 'w'))
consumers.append(p5)

time.sleep(4)
# Notice it gets assigned Partitions 0, 1, AND 2 simultaneously to do all the work itself!
!docker exec kafka kafka-consumer-groups --bootstrap-server localhost:9092 --describe --group group-B

---

## 🧹 Clean Up

First, we will politely ask all our background Python scripts to terminate.

In [ ]:
for p in consumers:
    p.terminate()
print("All background consumers successfully terminated.")

Then, stop the cluster and remove volumes to clean up the disk space.

In [ ]:
!docker-compose -f ../../docker-compose.yml down -v

<div style="background-color: rgba(102, 126, 234, 0.1); border: 1px solid rgba(102, 126, 234, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #667eea; margin-bottom: 10px;">🎉 Lab 8 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've successfully mastered Kafka Consumer Groups! You used a real Python application running natively on your machine to scale horizontally, demonstrated load balancing across log files, and proved the rules of partition assignment!</p>
</div>